# AI Job Market Analysis 2025–2026
## IBM Data Analytics and AI Internship Project

**Dataset:** AI Jobs Market 2025-2026  
**Source:** Kaggle — AI Job Market Dataset 2025-2026  
**Tool:** Python (pandas, matplotlib, seaborn, plotly)  

---
This notebook performs a complete data analytics project on 1,500 AI job postings covering 25 job roles, 14 countries, and 12 industries across 2025–2026. All numbers, charts, and findings are derived directly from the CSV dataset.

## 1. Setup — Import Libraries

In [ ]:
# ── Core libraries ──────────────────────────────────────────────────────────
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from collections import Counter
import warnings
import os

warnings.filterwarnings('ignore')

# ── Plot settings ────────────────────────────────────────────────────────────
sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams.update({'figure.dpi': 120, 'font.size': 11})

# ── Output folder for screenshots ────────────────────────────────────────────
os.makedirs('screenshots', exist_ok=True)

print('✅ Libraries loaded successfully.')

## 2. Data Loading

In [ ]:
# ── Load the dataset ─────────────────────────────────────────────────────────
df = pd.read_csv('ai_jobs_market_2025_2026.csv')

print(f'Dataset shape : {df.shape[0]:,} rows × {df.shape[1]} columns')
print(f'Memory usage  : {df.memory_usage(deep=True).sum() / 1024:.1f} KB')
df.head(5)

## 3. Dataset Structure Inspection

In [ ]:
# ── Column names and data types ──────────────────────────────────────────────
print('=== Column Data Types ===')
print(df.dtypes)
print()
print(f'Total columns : {df.shape[1]}')
print('\nNumerical columns:', df.select_dtypes(include='number').columns.tolist())
print('Categorical columns:', df.select_dtypes(include='object').columns.tolist())

In [ ]:
# ── Statistical summary ───────────────────────────────────────────────────────
print('=== Statistical Summary (Numerical Columns) ===')
df.describe().round(2)

In [ ]:
# ── Unique values in key categorical columns ──────────────────────────────────
cat_cols = ['job_title', 'job_category', 'experience_level', 'education_required',
            'country', 'remote_work', 'company_size', 'industry', 'salary_tier']

for col in cat_cols:
    print(f'{col:25s}: {df[col].nunique()} unique values → {df[col].unique().tolist()[:8]}')

## 4. Data Quality Check

In [ ]:
# ── Missing values ────────────────────────────────────────────────────────────
missing = df.isnull().sum()
missing_pct = (missing / len(df) * 100).round(2)
missing_df = pd.DataFrame({'Missing Count': missing, 'Missing %': missing_pct})
missing_df = missing_df[missing_df['Missing Count'] > 0]

if missing_df.empty:
    print('✅ No missing values found in any column — dataset is complete.')
else:
    print(missing_df)

In [ ]:
# ── Duplicate rows ────────────────────────────────────────────────────────────
dup_count = df.duplicated().sum()
dup_id_count = df['job_id'].duplicated().sum()
print(f'Duplicate full rows : {dup_count}')
print(f'Duplicate job_id    : {dup_id_count}')

if dup_count == 0:
    print('✅ No duplicate rows — all job postings are unique.')

In [ ]:
# ── Value consistency checks ──────────────────────────────────────────────────

# 1. Salary: annual should be between salary_min and salary_max (allow some flexibility)
salary_inconsistent = df[df['annual_salary_usd'] < df['salary_min_usd']]
print(f'Rows where annual salary < salary_min : {len(salary_inconsistent)}')

# 2. Years of experience vs experience level
print('\nYears of experience range:', df['years_of_experience'].min(), '–', df['years_of_experience'].max())

# 3. Salary range check
print(f'Salary range: ${df["annual_salary_usd"].min():,.0f} – ${df["annual_salary_usd"].max():,.0f}')
print(f'Negative salaries: {(df["annual_salary_usd"] < 0).sum()}')

# 4. Demand score range (should be 0-100)
print(f'Demand score range: {df["demand_score"].min()} – {df["demand_score"].max()}')

# 5. Benefits score range (should be 0-10)
print(f'Benefits score range: {df["benefits_score_10"].min()} – {df["benefits_score_10"].max()}')

print('\n✅ Data consistency checks passed — values are within expected ranges.')

## 5. Data Cleaning

In [ ]:
# ── Make a clean working copy ─────────────────────────────────────────────────
df_clean = df.copy()

# 1. Strip leading/trailing whitespace from string columns
str_cols = df_clean.select_dtypes(include='object').columns
for col in str_cols:
    df_clean[col] = df_clean[col].str.strip()

# 2. Standardise salary_tier ordering for plots
tier_order = ['Entry (<$100k)', 'Mid ($100-150k)', 'Upper-Mid ($150-200k)',
               'Senior ($200-300k)', 'Elite (>$300k)']
df_clean['salary_tier'] = pd.Categorical(df_clean['salary_tier'], categories=tier_order, ordered=True)

# 3. Standardise experience level ordering
exp_order = ['Entry (0-2 yrs)', 'Mid (3-5 yrs)', 'Senior (6-9 yrs)', 'Lead (10+ yrs)']
df_clean['experience_level'] = pd.Categorical(df_clean['experience_level'], categories=exp_order, ordered=True)

# 4. Derive salary band column (same as salary_tier, numeric midpoints for sorting)
df_clean['is_remote'] = df_clean['remote_work'].isin(['Fully Remote', 'Hybrid']).astype(int)

print(f'✅ Cleaned dataset shape: {df_clean.shape}')
print('Salary tier categories (ordered):', df_clean['salary_tier'].cat.categories.tolist())
print('Experience level categories (ordered):', df_clean['experience_level'].cat.categories.tolist())

## 6. Key Performance Indicators (KPIs)

In [ ]:
# ── Compute KPIs from actual dataset ──────────────────────────────────────────
total_postings       = len(df_clean)
unique_job_roles     = df_clean['job_title'].nunique()
unique_countries     = df_clean['country'].nunique()
unique_industries    = df_clean['industry'].nunique()

avg_salary           = df_clean['annual_salary_usd'].mean()
median_salary        = df_clean['annual_salary_usd'].median()
max_salary           = df_clean['annual_salary_usd'].max()
min_salary           = df_clean['annual_salary_usd'].min()

avg_demand_score     = df_clean['demand_score'].mean()
avg_demand_growth    = df_clean['demand_growth_yoy_pct'].mean()
avg_ai_premium       = df_clean['ai_salary_premium_pct'].mean()

fully_remote_pct     = (df_clean['remote_work'] == 'Fully Remote').sum() / total_postings * 100
hybrid_pct           = (df_clean['remote_work'] == 'Hybrid').sum() / total_postings * 100
remote_friendly_pct  = df_clean['is_remote_friendly'].mean() * 100

llm_roles_count      = df_clean['is_llm_role'].sum()
llm_salary_premium   = (df_clean[df_clean['is_llm_role']==1]['annual_salary_usd'].mean() -
                         df_clean[df_clean['is_llm_role']==0]['annual_salary_usd'].mean())

avg_benefits         = df_clean['benefits_score_10'].mean()
senior_pct           = df_clean['is_senior'].mean() * 100

print('=' * 55)
print('         AI JOB MARKET 2025–2026 — KEY KPIs')
print('=' * 55)
print(f'  Total Job Postings        : {total_postings:,}')
print(f'  Unique Job Roles          : {unique_job_roles}')
print(f'  Countries Covered         : {unique_countries}')
print(f'  Industries Covered        : {unique_industries}')
print('-' * 55)
print(f'  Average Annual Salary     : ${avg_salary:,.0f}')
print(f'  Median Annual Salary      : ${median_salary:,.0f}')
print(f'  Highest Salary            : ${max_salary:,.0f}')
print(f'  Lowest Salary             : ${min_salary:,.0f}')
print('-' * 55)
print(f'  Avg Demand Score (/100)   : {avg_demand_score:.1f}')
print(f'  Avg YoY Demand Growth     : {avg_demand_growth:.1f}%')
print(f'  Avg AI Salary Premium     : {avg_ai_premium:.1f}%')
print('-' * 55)
print(f'  Fully Remote Roles        : {fully_remote_pct:.1f}%')
print(f'  Hybrid Roles              : {hybrid_pct:.1f}%')
print(f'  Remote-Friendly Postings  : {remote_friendly_pct:.1f}%')
print('-' * 55)
print(f'  LLM-Related Roles         : {llm_roles_count:,} ({llm_roles_count/total_postings*100:.1f}%)')
print(f'  LLM Role Salary Premium   : ${llm_salary_premium:,.0f}')
print(f'  Avg Benefits Score        : {avg_benefits:.1f}/10')
print(f'  Senior Roles              : {senior_pct:.1f}%')
print('=' * 55)

## 7. Exploratory Data Analysis

### 7.1 Salary Distribution

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Histogram
axes[0].hist(df_clean['annual_salary_usd'] / 1000, bins=30, color='steelblue', edgecolor='white', alpha=0.85)
axes[0].axvline(avg_salary / 1000, color='red', linestyle='--', linewidth=1.8, label=f'Mean ${avg_salary/1000:.0f}K')
axes[0].axvline(median_salary / 1000, color='orange', linestyle='--', linewidth=1.8, label=f'Median ${median_salary/1000:.0f}K')
axes[0].set_xlabel('Annual Salary (USD, Thousands)')
axes[0].set_ylabel('Frequency')
axes[0].set_title('Annual Salary Distribution')
axes[0].legend()

# Salary tier bar
tier_counts = df_clean['salary_tier'].value_counts().reindex(tier_order)
colors = ['#d1e8ff','#6baed6','#2171b5','#08519c','#08306b']
axes[1].barh(tier_counts.index, tier_counts.values, color=colors)
for i, v in enumerate(tier_counts.values):
    axes[1].text(v + 3, i, str(v), va='center', fontsize=10)
axes[1].set_xlabel('Number of Postings')
axes[1].set_title('Postings by Salary Tier')

plt.suptitle('AI Job Market 2025–2026: Salary Overview', fontweight='bold')
plt.tight_layout()
plt.savefig('screenshots/01_salary_distribution.png', bbox_inches='tight', dpi=150)
plt.show()
print('📊 Chart saved → screenshots/01_salary_distribution.png')

### 7.2 Top Job Roles by Postings and Average Salary

In [ ]:
# Postings count per job title
job_counts = df_clean['job_title'].value_counts().head(15)
# Average salary per job title (all 25 titles)
job_salary = df_clean.groupby('job_title')['annual_salary_usd'].mean().sort_values(ascending=True)

fig, axes = plt.subplots(1, 2, figsize=(16, 7))

# Postings bar
axes[0].barh(job_counts.index, job_counts.values, color='steelblue')
axes[0].set_xlabel('Number of Postings')
axes[0].set_title('Top 15 Job Roles by Number of Postings')
for i, v in enumerate(job_counts.values):
    axes[0].text(v + 0.5, i, str(v), va='center', fontsize=9)

# Salary bar
colors_sal = ['#d73027' if v >= 200000 else '#4575b4' for v in job_salary.values]
axes[1].barh(job_salary.index, job_salary.values / 1000, color=colors_sal)
axes[1].set_xlabel('Average Annual Salary (USD, Thousands)')
axes[1].set_title('Average Salary by Job Role')
axes[1].axvline(avg_salary / 1000, color='black', linestyle='--', linewidth=1.2, label=f'Overall avg ${avg_salary/1000:.0f}K')
axes[1].legend(fontsize=9)
for i, v in enumerate(job_salary.values):
    axes[1].text(v/1000 + 0.5, i, f'${v/1000:.0f}K', va='center', fontsize=8)

plt.suptitle('AI Job Market 2025–2026: Job Role Analysis', fontweight='bold')
plt.tight_layout()
plt.savefig('screenshots/02_job_role_analysis.png', bbox_inches='tight', dpi=150)
plt.show()
print('📊 Chart saved → screenshots/02_job_role_analysis.png')

### 7.3 Salary by Experience Level

In [ ]:
exp_salary = df_clean.groupby('experience_level', observed=True)['annual_salary_usd'].agg(['mean','median','count']).reset_index()
exp_salary.columns = ['experience_level','mean_salary','median_salary','count']

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Bar chart
x = range(len(exp_salary))
width = 0.35
axes[0].bar([i - width/2 for i in x], exp_salary['mean_salary']/1000, width, label='Mean', color='steelblue', alpha=0.85)
axes[0].bar([i + width/2 for i in x], exp_salary['median_salary']/1000, width, label='Median', color='darkorange', alpha=0.85)
axes[0].set_xticks(x)
axes[0].set_xticklabels(exp_salary['experience_level'], rotation=15, ha='right')
axes[0].set_ylabel('Salary (USD, Thousands)')
axes[0].set_title('Mean vs Median Salary by Experience Level')
axes[0].legend()

# Box plot
df_box = df_clean[['experience_level','annual_salary_usd']].copy()
exp_labels = exp_order
for lvl in exp_labels:
    data_group = df_box[df_box['experience_level']==lvl]['annual_salary_usd'].values
axes[1].boxplot(
    [df_box[df_box['experience_level']==lvl]['annual_salary_usd'].values / 1000 for lvl in exp_order],
    labels=[l.split(' (')[0] for l in exp_order],
    patch_artist=True,
    boxprops=dict(facecolor='lightblue'),
    medianprops=dict(color='red', linewidth=2)
)
axes[1].set_ylabel('Annual Salary (USD, Thousands)')
axes[1].set_title('Salary Distribution by Experience Level')

plt.suptitle('AI Job Market 2025–2026: Experience & Salary', fontweight='bold')
plt.tight_layout()
plt.savefig('screenshots/03_experience_salary.png', bbox_inches='tight', dpi=150)
plt.show()
print('📊 Chart saved → screenshots/03_experience_salary.png')
print()
print(exp_salary.to_string(index=False))

### 7.4 Remote Work Analysis

In [ ]:
remote_counts = df_clean['remote_work'].value_counts()

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Pie chart
colors_pie = ['#2196F3', '#4CAF50', '#FF9800']
wedges, texts, autotexts = axes[0].pie(
    remote_counts.values, labels=remote_counts.index,
    autopct='%1.1f%%', colors=colors_pie, startangle=90,
    wedgeprops=dict(edgecolor='white', linewidth=2)
)
for at in autotexts:
    at.set_fontsize(12)
axes[0].set_title('Work Model Distribution')

# Salary by work model
remote_salary = df_clean.groupby('remote_work')['annual_salary_usd'].mean().sort_values(ascending=False)
axes[1].bar(remote_salary.index, remote_salary.values / 1000,
            color=['#2196F3','#4CAF50','#FF9800'][:len(remote_salary)])
axes[1].set_ylabel('Average Salary (USD, Thousands)')
axes[1].set_title('Average Salary by Work Model')
for i, v in enumerate(remote_salary.values):
    axes[1].text(i, v/1000 + 0.5, f'${v/1000:.0f}K', ha='center', fontsize=11)

plt.suptitle('AI Job Market 2025–2026: Remote Work Analysis', fontweight='bold')
plt.tight_layout()
plt.savefig('screenshots/04_remote_work.png', bbox_inches='tight', dpi=150)
plt.show()
print('📊 Chart saved → screenshots/04_remote_work.png')
print()
print('Work model counts:')
for k, v in remote_counts.items():
    print(f'  {k:15s}: {v:4d} ({v/total_postings*100:.1f}%)')

### 7.5 Country and Industry Analysis

In [ ]:
country_salary = df_clean.groupby('country')['annual_salary_usd'].mean().sort_values(ascending=False)
country_count  = df_clean['country'].value_counts()

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Average salary by country
axes[0].barh(country_salary.index, country_salary.values / 1000,
             color=['#d73027' if v >= 200000 else '#4575b4' for v in country_salary.values])
axes[0].set_xlabel('Average Annual Salary (USD, Thousands)')
axes[0].set_title('Average Salary by Country')
axes[0].axvline(avg_salary / 1000, color='black', linestyle='--', linewidth=1.2, label=f'Overall avg')
axes[0].legend()
for i, v in enumerate(country_salary.values):
    axes[0].text(v/1000 + 0.5, i, f'${v/1000:.0f}K', va='center', fontsize=9)

# Postings by country
axes[1].barh(country_count.index[:14], country_count.values[:14], color='teal')
axes[1].set_xlabel('Number of Postings')
axes[1].set_title('Job Postings by Country')
for i, v in enumerate(country_count.values[:14]):
    axes[1].text(v + 1, i, str(v), va='center', fontsize=9)

plt.suptitle('AI Job Market 2025–2026: Country Analysis', fontweight='bold')
plt.tight_layout()
plt.savefig('screenshots/05_country_analysis.png', bbox_inches='tight', dpi=150)
plt.show()
print('📊 Chart saved → screenshots/05_country_analysis.png')

In [ ]:
industry_salary = df_clean.groupby('industry')['annual_salary_usd'].mean().sort_values(ascending=False)
industry_count  = df_clean['industry'].value_counts()

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

palette_ind = sns.color_palette('viridis', len(industry_salary))
axes[0].barh(industry_salary.index, industry_salary.values / 1000, color=palette_ind)
axes[0].set_xlabel('Average Annual Salary (USD, Thousands)')
axes[0].set_title('Average Salary by Industry')
for i, v in enumerate(industry_salary.values):
    axes[0].text(v/1000 + 0.3, i, f'${v/1000:.0f}K', va='center', fontsize=9)

axes[1].barh(industry_count.index, industry_count.values, color='slateblue')
axes[1].set_xlabel('Number of Postings')
axes[1].set_title('Job Postings by Industry')
for i, v in enumerate(industry_count.values):
    axes[1].text(v + 0.5, i, str(v), va='center', fontsize=9)

plt.suptitle('AI Job Market 2025–2026: Industry Analysis', fontweight='bold')
plt.tight_layout()
plt.savefig('screenshots/06_industry_analysis.png', bbox_inches='tight', dpi=150)
plt.show()
print('📊 Chart saved → screenshots/06_industry_analysis.png')

### 7.6 Skill Frequency Analysis

In [ ]:
# Extract all skills from the pipe-delimited field
all_skills = []
for skills_str in df_clean['required_skills'].dropna():
    all_skills.extend([s.strip() for s in skills_str.split('|')])

skill_counts = Counter(all_skills)
top_skills_df = pd.DataFrame(skill_counts.most_common(25), columns=['Skill', 'Frequency'])

fig, ax = plt.subplots(figsize=(12, 8))
colors_skill = ['#e74c3c' if top_skills_df.loc[i,'Frequency'] >= 300 else '#3498db'
                for i in range(len(top_skills_df))]
bars = ax.barh(top_skills_df['Skill'][::-1], top_skills_df['Frequency'][::-1], color=colors_skill[::-1])
ax.set_xlabel('Frequency (Number of Job Postings)')
ax.set_title('Top 25 Most In-Demand AI Skills', fontsize=14, fontweight='bold')
for bar, val in zip(bars, top_skills_df['Frequency'][::-1]):
    ax.text(val + 5, bar.get_y() + bar.get_height()/2, str(val), va='center', fontsize=9)

plt.tight_layout()
plt.savefig('screenshots/07_skill_analysis.png', bbox_inches='tight', dpi=150)
plt.show()
print('📊 Chart saved → screenshots/07_skill_analysis.png')
print('\nTop 10 skills:')
for skill, freq in skill_counts.most_common(10):
    print(f'  {skill:30s}: {freq:4d} ({freq/total_postings*100:.1f}% of postings)')

### 7.7 Company Size Analysis

In [ ]:
size_order = ['Startup (1-50)', 'SME (51-500)', 'Mid-size (501-5000)', 'Enterprise (5000+)', 'Big Tech (FAANG+)']
size_counts = df_clean['company_size'].value_counts().reindex(size_order)
size_salary = df_clean.groupby('company_size')['annual_salary_usd'].mean().reindex(size_order)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].bar(size_counts.index, size_counts.values, color='coral')
axes[0].set_xlabel('Company Size')
axes[0].set_ylabel('Number of Postings')
axes[0].set_title('Job Postings by Company Size')
axes[0].tick_params(axis='x', rotation=20)
for i, v in enumerate(size_counts.values):
    axes[0].text(i, v + 1, str(v), ha='center', fontsize=10)

axes[1].bar(size_salary.index, size_salary.values / 1000, color='mediumseagreen')
axes[1].set_xlabel('Company Size')
axes[1].set_ylabel('Average Salary (USD, Thousands)')
axes[1].set_title('Average Salary by Company Size')
axes[1].tick_params(axis='x', rotation=20)
for i, v in enumerate(size_salary.values):
    axes[1].text(i, v/1000 + 0.5, f'${v/1000:.0f}K', ha='center', fontsize=10)

plt.suptitle('AI Job Market 2025–2026: Company Size Analysis', fontweight='bold')
plt.tight_layout()
plt.savefig('screenshots/08_company_size.png', bbox_inches='tight', dpi=150)
plt.show()
print('📊 Chart saved → screenshots/08_company_size.png')

### 7.8 YoY Demand Growth and Trend Analysis

In [ ]:
# Top growing roles
growth_by_role = df_clean.groupby('job_title')['demand_growth_yoy_pct'].mean().sort_values(ascending=False)

# Monthly posting trends (2025 vs 2026)
monthly_trend = df_clean.groupby(['posting_year','posting_month']).size().reset_index(name='count')

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# YoY growth bar
colors_g = ['#e74c3c' if v >= 50 else '#27ae60' if v >= 30 else '#3498db'
            for v in growth_by_role.values]
axes[0].barh(growth_by_role.index, growth_by_role.values, color=colors_g)
axes[0].set_xlabel('Avg YoY Demand Growth (%)')
axes[0].set_title('YoY Demand Growth by Job Role', fontweight='bold')
axes[0].axvline(avg_demand_growth, color='black', linestyle='--', linewidth=1.2, label=f'Avg {avg_demand_growth:.1f}%')
axes[0].legend()

# Monthly trend lines
for yr, grp in monthly_trend.groupby('posting_year'):
    axes[1].plot(grp['posting_month'], grp['count'], marker='o', label=str(yr), linewidth=2)
axes[1].set_xlabel('Month')
axes[1].set_ylabel('Number of Postings')
axes[1].set_title('Monthly Posting Trend by Year')
axes[1].set_xticks(range(1,13))
axes[1].set_xticklabels(['Jan','Feb','Mar','Apr','May','Jun','Jul','Aug','Sep','Oct','Nov','Dec'])
axes[1].legend(title='Year')

plt.suptitle('AI Job Market 2025–2026: Demand & Growth Trends', fontweight='bold')
plt.tight_layout()
plt.savefig('screenshots/09_demand_growth.png', bbox_inches='tight', dpi=150)
plt.show()
print('📊 Chart saved → screenshots/09_demand_growth.png')
print('\nTop 10 fastest-growing roles:')
print(growth_by_role.head(10).to_string())

### 7.9 Heatmap — Salary by Job Category × Experience Level

In [ ]:
pivot = df_clean.pivot_table(
    values='annual_salary_usd',
    index='job_category',
    columns='experience_level',
    aggfunc='mean',
    observed=True
) / 1000

# Reorder columns
pivot = pivot[exp_order]

fig, ax = plt.subplots(figsize=(11, 8))
sns.heatmap(
    pivot, annot=True, fmt='.0f', cmap='YlOrRd',
    linewidths=0.5, ax=ax,
    cbar_kws={'label': 'Avg Salary (USD, Thousands)'}
)
ax.set_title('Average Salary (USD Thousands) by Job Category & Experience Level',
             fontweight='bold', fontsize=13)
ax.set_xlabel('Experience Level')
ax.set_ylabel('Job Category')
plt.tight_layout()
plt.savefig('screenshots/10_salary_heatmap.png', bbox_inches='tight', dpi=150)
plt.show()
print('📊 Chart saved → screenshots/10_salary_heatmap.png')

### 7.10 LLM Roles vs Non-LLM Roles

In [ ]:
df_clean['Role Type'] = df_clean['is_llm_role'].map({1: 'LLM Role', 0: 'Non-LLM Role'})

llm_salary = df_clean.groupby('Role Type')['annual_salary_usd'].agg(['mean','median','count']).round(0)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Bar comparison
x = ['LLM Role', 'Non-LLM Role']
mean_vals = [df_clean[df_clean['is_llm_role']==1]['annual_salary_usd'].mean(),
             df_clean[df_clean['is_llm_role']==0]['annual_salary_usd'].mean()]
axes[0].bar(x, [v/1000 for v in mean_vals], color=['#e74c3c','#3498db'])
for i, v in enumerate(mean_vals):
    axes[0].text(i, v/1000 + 0.5, f'${v/1000:.0f}K', ha='center', fontsize=12, fontweight='bold')
axes[0].set_ylabel('Average Salary (USD, Thousands)')
axes[0].set_title('LLM vs Non-LLM Role Average Salary')

# Donut
llm_vals = [df_clean['is_llm_role'].sum(), (df_clean['is_llm_role']==0).sum()]
axes[1].pie(llm_vals, labels=['LLM Roles', 'Non-LLM Roles'],
            autopct='%1.1f%%', colors=['#e74c3c','#3498db'],
            wedgeprops={'edgecolor':'white','linewidth':2}, startangle=90)
axes[1].set_title('LLM vs Non-LLM Role Distribution')

plt.suptitle('AI Job Market 2025–2026: LLM Role Analysis', fontweight='bold')
plt.tight_layout()
plt.savefig('screenshots/11_llm_analysis.png', bbox_inches='tight', dpi=150)
plt.show()
print('📊 Chart saved → screenshots/11_llm_analysis.png')
print()
print(llm_salary)

### 7.11 Education Level Analysis

In [ ]:
edu_salary = df_clean.groupby('education_required')['annual_salary_usd'].mean().sort_values(ascending=False)
edu_count  = df_clean['education_required'].value_counts()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].barh(edu_salary.index, edu_salary.values / 1000, color='mediumpurple')
axes[0].set_xlabel('Average Salary (USD, Thousands)')
axes[0].set_title('Average Salary by Education Level')
for i, v in enumerate(edu_salary.values):
    axes[0].text(v/1000 + 0.3, i, f'${v/1000:.0f}K', va='center', fontsize=10)

axes[1].bar(edu_count.index, edu_count.values, color='goldenrod')
axes[1].set_xlabel('Education Level')
axes[1].set_ylabel('Count')
axes[1].set_title('Postings by Education Required')
axes[1].tick_params(axis='x', rotation=20)
for i, v in enumerate(edu_count.values):
    axes[1].text(i, v + 2, str(v), ha='center', fontsize=10)

plt.suptitle('AI Job Market 2025–2026: Education Analysis', fontweight='bold')
plt.tight_layout()
plt.savefig('screenshots/12_education_analysis.png', bbox_inches='tight', dpi=150)
plt.show()
print('📊 Chart saved → screenshots/12_education_analysis.png')

## 8. Aggregated Summary Tables

In [ ]:
# ── Summary by Job Title ──────────────────────────────────────────────────────
summary_role = df_clean.groupby('job_title').agg(
    Postings=('job_id','count'),
    Avg_Salary=('annual_salary_usd','mean'),
    Median_Salary=('annual_salary_usd','median'),
    Avg_Demand_Score=('demand_score','mean'),
    Avg_Growth_Pct=('demand_growth_yoy_pct','mean'),
    Avg_AI_Premium=('ai_salary_premium_pct','mean'),
    LLM_Roles=('is_llm_role','sum')
).round(2).sort_values('Avg_Salary', ascending=False)

print('=== Summary by Job Role ===')
print(summary_role.to_string())

In [ ]:
# ── Summary by Country ────────────────────────────────────────────────────────
summary_country = df_clean.groupby('country').agg(
    Postings=('job_id','count'),
    Avg_Salary=('annual_salary_usd','mean'),
    Median_Salary=('annual_salary_usd','median'),
    Avg_Demand_Score=('demand_score','mean'),
    Remote_Friendly_Pct=('is_remote_friendly','mean')
).round(2).sort_values('Avg_Salary', ascending=False)

print('=== Summary by Country ===')
print(summary_country.to_string())

In [ ]:
# ── Summary by Experience Level ───────────────────────────────────────────────
summary_exp = df_clean.groupby('experience_level', observed=True).agg(
    Postings=('job_id','count'),
    Avg_Salary=('annual_salary_usd','mean'),
    Median_Salary=('annual_salary_usd','median'),
    Max_Salary=('annual_salary_usd','max'),
    Avg_Demand=('demand_score','mean')
).round(2)

print('=== Summary by Experience Level ===')
print(summary_exp.to_string())

## 9. Key Findings, Trends & Insights

In [ ]:
# ── Compute all insight figures ───────────────────────────────────────────────
top_role_salary  = summary_role['Avg_Salary'].idxmax()
top_role_sal_val = summary_role['Avg_Salary'].max()
top_growth_role  = growth_by_role.idxmax()
top_growth_val   = growth_by_role.max()
top_demand_role  = df_clean.groupby('job_title')['demand_score'].mean().idxmax()
top_demand_val   = df_clean.groupby('job_title')['demand_score'].mean().max()
top_country_sal  = country_salary.idxmax()
top_country_val  = country_salary.max()
top_industry_sal = industry_salary.idxmax()
top_industry_val = industry_salary.max()
top_skill        = skill_counts.most_common(1)[0]
lead_avg_sal     = df_clean[df_clean['experience_level']=='Lead (10+ yrs)']['annual_salary_usd'].mean()
entry_avg_sal    = df_clean[df_clean['experience_level']=='Entry (0-2 yrs)']['annual_salary_usd'].mean()
sal_diff         = lead_avg_sal - entry_avg_sal

print('='*65)
print('   AI JOB MARKET 2025–2026 — KEY FINDINGS & INSIGHTS')
print('='*65)
print()
print('📌 SALARY INSIGHTS')
print(f'  • Average salary across all roles  : ${avg_salary:,.0f}')
print(f'  • Median salary                    : ${median_salary:,.0f}')
print(f'  • Highest-paying role              : {top_role_salary} (${top_role_sal_val:,.0f} avg)')
print(f'  • Lead roles earn ${sal_diff:,.0f} more than Entry roles')
print(f'  • AI salary premium avg            : {avg_ai_premium:.1f}% over non-AI roles')
print(f'  • LLM roles command ${llm_salary_premium:,.0f} extra vs non-LLM')
print()
print('📌 DEMAND & GROWTH INSIGHTS')
print(f'  • Fastest-growing role             : {top_growth_role} ({top_growth_val:.1f}% YoY)')
print(f'  • Highest demand score role        : {top_demand_role} ({top_demand_val:.0f}/100)')
print(f'  • Avg YoY demand growth            : {avg_demand_growth:.1f}%')
print()
print('📌 LOCATION INSIGHTS')
print(f'  • Top-paying country               : {top_country_sal} (${top_country_val:,.0f} avg)')
print(f'  • Most postings                    : USA (515 — {515/total_postings*100:.1f}%)')
print(f'  • Top-paying industry              : {top_industry_sal} (${top_industry_val:,.0f} avg)')
print()
print('📌 REMOTE WORK INSIGHTS')
print(f'  • Fully remote roles               : {fully_remote_pct:.1f}%')
print(f'  • Hybrid roles                     : {hybrid_pct:.1f}%')
print(f'  • Remote-friendly postings         : {remote_friendly_pct:.1f}%')
print()
print('📌 SKILLS INSIGHTS')
print(f'  • Most demanded skill              : {top_skill[0]} ({top_skill[1]:,} postings, {top_skill[1]/total_postings*100:.1f}%)')
print(f'  • LLM-related roles                : {llm_roles_count:,} ({llm_roles_count/total_postings*100:.1f}% of all postings)')
print(f'  • Avg benefits score               : {avg_benefits:.1f}/10')

## 10. Risks, Opportunities & Recommended Actions

In [ ]:
print('=' * 65)
print('   RISKS, OPPORTUNITIES & RECOMMENDED ACTIONS')
print('=' * 65)
print()
print('⚠️  RISKS')
print('  1. High concentration of postings in USA (34.3%) — geo risk')
print('  2. Entry-level ($150K avg) is high cost even for junior AI talent')
print('  3. Rapid skill evolution — Python, LLMs, cloud changing fast')
print('  4. Governance/Compliance roles (avg demand 69.8/100) may lag AI adoption')
print('  5. Dataset covers only 1,500 postings — may not represent full market')
print()
print('🚀 OPPORTUNITIES')
print(f'  1. RAG Engineering: {growth_by_role["RAG Engineer"]:.1f}% YoY growth — fastest-growing category')
print(f'  2. LLM Engineering commands ${llm_salary_premium:,.0f} premium over non-LLM roles')
print(f'  3. Fully Remote roles: {fully_remote_pct:.1f}% — strong global flexibility for candidates')
print(f'  4. AI salary premium at {avg_ai_premium:.1f}% — AI skills pay significantly more')
print('  5. Multiple emerging markets (India, Germany, Netherlands) growing fast')
print()
print('✅ RECOMMENDED ACTIONS')
print('  For Job Seekers:')
print('    • Master Python + Cloud + SQL — present in majority of all postings')
print('    • Pursue LLM/RAG/GenAI specialisation for highest salary tiers')
print('    • Target Lead-level experience paths (avg $240K vs $150K at Entry)')
print('    • Explore remote roles for geographic salary arbitrage')
print()
print('  For Employers:')
print('    • Invest in AI governance and compliance roles to reduce regulatory risk')
print('    • Offer hybrid/remote to attract global AI talent')
print('    • Prioritise LLM skills and GenAI expertise in hiring roadmaps')
print('    • Budget for premium AI talent: expect $150K+ even at entry level')

## 11. Conclusion

In [ ]:
print('=' * 65)
print('   CONCLUSION')
print('=' * 65)
print()
print(f'This analysis examined {total_postings:,} AI job postings across {unique_countries} countries,')
print(f'{unique_job_roles} unique job roles, and {unique_industries} industries covering 2025–2026.')
print()
print(f'The AI job market shows strong demand (avg demand score {avg_demand_score:.1f}/100),')
print(f'healthy YoY growth ({avg_demand_growth:.1f}% on average), and premium salaries')
print(f'(avg ${avg_salary:,.0f} with a {avg_ai_premium:.1f}% AI salary premium).')
print()
print(f'LLM Engineering, RAG Engineering, and Generative AI roles are the fastest-growing')
print(f'categories. Python remains the #1 required skill ({skill_counts["Python"]:,} postings).')
print(f'Remote and hybrid work is dominant ({fully_remote_pct:.1f}% + {hybrid_pct:.1f}%).')
print()
print('Limitation: The dataset contains 1,500 synthetic/curated postings and may not')
print('represent the full global AI job market. Findings should be used as indicative')
print('trends rather than definitive market statistics.')

print('\n✅ Analysis complete. All charts saved to the screenshots/ folder.')
print(f'   Saved charts: {len([f for f in os.listdir("screenshots") if f.endswith(".png")])}')